In [1]:
import numpy as np

def load_letters(path):
    letters = []

    with open(path, encoding="utf8") as f:
        for line in f:
            token = line.strip()
            if token:
                letters.append(token)

    return letters


def build_ctc_vocab(letters):

    charset = set(letters)

    # базовые пробел и апостроф
    charset.add(" ")
    charset.add("'")

    # важно: сортировка фиксирует воспроизводимость
    charset = sorted(charset)

    # добавляем OOV токен (важно для IAM robustness)
    char2id = {
        "<blank>": 0,
        "<unk>": 1
    }

    for idx, ch in enumerate(charset, start=2):
        char2id[ch] = idx

    id2char = {v: k for k, v in char2id.items()}

    return char2id, id2char

In [2]:
from pathlib import Path

SPECIAL_TOKENS = {
    "sp": " ",
    "ga": "'"
}

def read_mlf(path):
    labels = {}

    current_id = None
    chars = []

    with open(path, encoding="utf8") as f:
        for line in f:
            line = line.strip()

            if not line or line == "#!MLF!#":
                continue

            if line.startswith('"'):
                if current_id is not None:
                    labels[current_id] = "".join(chars)

                current_id = Path(line.strip('"')).stem
                chars = []
                continue

            if line == ".":
                continue

            chars.append(SPECIAL_TOKENS.get(line, line))

    if current_id is not None:
        labels[current_id] = "".join(chars)

    return labels

In [3]:
import numpy as np

def encode_text(text, char2id):
    return np.asarray(
        [char2id.get(ch, char2id["<unk>"]) for ch in text],
        dtype=np.int32
    )

In [4]:
import xml.etree.ElementTree as ET


def parse_strokes(xml_file):

    root = ET.parse(xml_file).getroot()

    strokes = []

    for stroke in root.iter("Stroke"):

        pts = []

        for p in stroke.iter("Point"):

            pts.append(
                (
                    float(p.attrib["x"]),
                    float(p.attrib["y"]),
                    float(
                        p.attrib.get("time", 0)
                    )
                )
            )

        strokes.append(pts)

    return strokes

In [5]:
import numpy as np


def normalize_strokes(strokes):

    all_pts = np.array([
        [x, y]
        for s in strokes
        for x, y, _ in s
    ])

    xmin, ymin = all_pts.min(axis=0)
    xmax, ymax = all_pts.max(axis=0)

    sx = xmax - xmin + 1e-8
    sy = ymax - ymin + 1e-8

    out = []

    for stroke in strokes:

        cur = []

        for x, y, t in stroke:

            cur.append(
                (
                    (x - xmin) / sx,
                    (y - ymin) / sy,
                    t
                )
            )

        out.append(cur)

    return out

In [6]:
import numpy as np

def build_nodes(strokes):
    all_times = [
        t
        for stroke in strokes
        for _, _, t in stroke
    ]

    t_min = min(all_times)
    t_max = max(all_times)

    t_scale = max(t_max - t_min, 1e-6)

    nodes = []

    prev_global_x = None
    prev_global_y = None
    prev_global_t = None

    for stroke_id, stroke in enumerate(strokes):

        prev_x = None
        prev_y = None
        prev_t = None

        for i, (x, y, t) in enumerate(stroke):

            # dx/dy ONLY inside stroke (FIXED)
            if prev_x is None:
                dx = 0.0
                dy = 0.0
                dt = 0.0
                vel = 0.0
            else:
                dx = x - prev_x
                dy = y - prev_y
                dt = max(
                    (t - prev_t) / t_scale,
                    1e-6
                )
                vel = (dx*dx + dy*dy) ** 0.5 / dt

            # cross-stroke continuity (optional signal)
            if prev_global_x is None:
                gdx = 0.0
                gdy = 0.0
            else:
                gdx = x - prev_global_x
                gdy = y - prev_global_y

            is_first_in_stroke = 1.0 if i == 0 else 0.0

            nodes.append([
                x,
                y,
                dx,
                dy,
                dt,
                vel,
                float(stroke_id),
                is_first_in_stroke
            ])

            prev_x, prev_y, prev_t = x, y, t
            prev_global_x, prev_global_y, prev_global_t = x, y, t

    return np.asarray(nodes, dtype=np.float32)

In [7]:
import numpy as np

def build_graph(strokes):

    edge_index = []
    edge_attr = []

    offset = 0
    previous_last = None

    for stroke in strokes:

        n = len(stroke)

        # self-loops
        for i in range(n):
            idx = offset + i
            edge_index.append([idx, idx])
            edge_attr.append([0, 0, 0])  # self-loop

        # sequential edges
        for i in range(n - 1):
            a = offset + i
            b = offset + i + 1

            edge_index.append([a, b])
            edge_attr.append([1, 0, 1])  # seq forward

            edge_index.append([b, a])
            edge_attr.append([1, 0, 1])  # seq backward

        # stroke transition edge
        first_node = offset
        last_node = offset + n - 1

        if previous_last is not None:
            edge_index.append([previous_last, first_node])
            edge_attr.append([0, 1, 1])  # pen-up transition

        previous_last = last_node
        offset += n

    return (
        np.asarray(edge_index, dtype=np.int32),
        np.asarray(edge_attr, dtype=np.float32)
    )


def build_adjacency(nodes, edge_index):

    n = len(nodes)

    adj = np.zeros(
        (n, n),
        dtype=np.float32
    )

    for src, dst in edge_index:
        adj[src, dst] = 1.0

    return adj

In [8]:
import re

def get_form_id(xml_stem: str) -> str:
    """
    Converts:
        a01-020x-01 -> a01-020x
        a01-020x    -> a01-020x
    """
    return re.sub(r"-\d+[a-z]?$", "", xml_stem)

from pathlib import Path
from collections import defaultdict

def index_lines_by_form(strokes_root):
    """
    Builds:
        form_id -> list of xml files
    """

    form_map = defaultdict(list)

    for xml_file in Path(strokes_root).rglob("*.xml"):

        form_id = get_form_id(xml_file.stem)
        form_map[form_id].append(xml_file)

    return form_map

def load_split(path):
    with open(path) as f:
        return set(
            line.strip()
            for line in f
            if line.strip()
        )

In [18]:
from tqdm import tqdm

def build_dataset(
    strokes_root,
    labels,
    char2id,
    split_file=None
):

    dataset = []

    # FORM-level split
    allowed_forms = None
    if split_file is not None:
        allowed_forms = load_split(split_file)

    # index ALL line files by form
    form_map = index_lines_by_form(strokes_root)

    for form_id, xml_files in tqdm(form_map.items()):

        # split filter (FORM-level!)
        if allowed_forms is not None and form_id not in allowed_forms:
            continue

        # each form has multiple lines
        for xml_file in xml_files:

            line_id = xml_file.stem  # full line id

            # label must exist for LINE id
            if line_id not in labels:
                continue

            strokes = parse_strokes(xml_file)
            strokes = normalize_strokes(strokes)

            nodes = build_nodes(strokes)
            time_idx = np.arange(
                len(nodes),
                dtype=np.int32
            )
            edge_index, edge_attr = build_graph(strokes)

            target = encode_text(
                labels[line_id],
                char2id
            )

            input_length = len(nodes)

            target_length = len(target)

            dataset.append({
                "id": line_id,
                "form_id": form_id,

                "nodes": nodes,
                "edge_index": edge_index,
                "edge_attr": edge_attr,

                "target": target,
                "text": labels[line_id],
                "time_idx": time_idx,

                "input_length": input_length,
                "target_length": target_length,
            })

    return dataset

In [10]:
letters = load_letters("Technical files/letters")
char2id, id2char = build_ctc_vocab(letters)
labels = read_mlf("Technical files/labels.mlf")

In [17]:
len(char2id)

61

In [11]:
def get_dataset(split_name):
    dataset = build_dataset(
        strokes_root = "lineStrokes",
        labels = labels,
        char2id = char2id,
        split_file = split_name
    )
    return dataset

In [19]:
train = get_dataset('Splits/trainset.txt')

100%|██████████| 1730/1730 [00:32<00:00, 53.88it/s]


In [20]:
val_1 = get_dataset('Splits/testset_v.txt')
val_2 = get_dataset('Splits/testset_t.txt')
test = get_dataset('Splits/testset_f.txt')


100%|██████████| 1730/1730 [00:21<00:00, 79.97it/s] 


In [21]:
import pickle

In [22]:
with open("train.pkl", "wb") as f:
    pickle.dump(train, f)

with open("val1.pkl", "wb") as f:
    pickle.dump(val_1, f)

with open("val2.pkl", "wb") as f:
    pickle.dump(val_2, f)

with open("test.pkl", "wb") as f:
    pickle.dump(test, f)